# GastroTwin on Google Colab

Reproduces *Gastric Polyp Detection without Gastric Annotations Using a Stomach Digital Twin* (BVM 2027).

Runtime → Change runtime type → **A100 GPU** with High-RAM. Results are written to Google Drive and every
trained model is cached, so after a disconnect you can run all cells again and training continues.

In [ ]:
!git clone -q https://github.com/Darshan0109/GastroTwin.git
%pip install -q -e GastroTwin

## Data
Kvasir-SEG is downloaded automatically. GastroVision must be downloaded once by hand
(https://osf.io/84e7f/ or the Google Drive link in https://github.com/DebeshJha/GastroVision)
and placed anywhere in your Google Drive, as a zip or as an extracted folder.
The cell below finds both datasets and the output folder by itself.

In [ ]:
import glob, os
from google.colab import drive
drive.mount('/content/drive')

def find(*patterns):
    for pat in patterns:
        hits = glob.glob(f'/content/drive/MyDrive/**/{pat}', recursive=True)
        if hits:
            return hits[0].rstrip('/')
    return None

KVASIR = find('kvasir-seg*.zip', 'Kvasir-SEG*.zip', 'Kvasir-SEG/')
if KVASIR is None:
    !wget -q https://datasets.simula.no/downloads/kvasir-seg.zip -O /content/kvasir-seg.zip
    KVASIR = '/content/kvasir-seg.zip'

GASTROVISION = find('GastroVision*.zip', 'GV.zip', 'GastroVision/')
assert GASTROVISION, 'GastroVision not found in your Drive: download it from https://osf.io/84e7f/ first.'

runs = glob.glob('/content/drive/MyDrive/**/runs/A_kvasir__unet_r34__s0.npz', recursive=True)
OUT = os.path.dirname(os.path.dirname(runs[0])) if runs else '/content/drive/MyDrive/gastrotwin_out'

print('KVASIR       =', KVASIR)
print('GASTROVISION =', GASTROVISION)
print('OUT          =', OUT)
print('trained models already in OUT:', len(glob.glob(f'{OUT}/runs/*.npz')), 'of 78')

## Option 1: rebuild the paper's tables from the published runs
Takes about a minute and needs no data or GPU. Tables and figures are saved to `OUT/paper_results`.

In [ ]:
!mkdir -p "$OUT" && cp -r GastroTwin/results "$OUT/paper_results"
!cd GastroTwin && python scripts/make_tables.py --results "$OUT/paper_results"

## Option 2: retrain everything
Renders the twin, trains all 78 models and writes tables and figures to `OUT`.

In [ ]:
!cd GastroTwin && python scripts/run_experiments.py --kvasir "$KVASIR" --gastrovision "$GASTROVISION" --data-dir /content/data --out "$OUT"